# Saving & Serving Models

The last stage of the workflow, deploy and inference. Save a trained model to a file, serve its predictions in production, and keep watching it.

This notebook comes from the [Saving & Serving Models](https://pabl0aranda.github.io/Atlas-ML/deployment/overview/) page on Atlas ML, an interactive guide to machine learning. Run the cells from top to bottom.

In [ ]:
%pip install --quiet skl2onnx onnxruntime fastapi httpx

## Example data

The code on the page assumes some data already exists. This cell creates it.

In [ ]:
from sklearn.datasets import make_moons

# Two interleaving half-moons, like the sandbox's Moons dataset.
X, y = make_moons(n_samples=300, noise=0.25, random_state=7)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

## Code from the page

In [ ]:
import joblib
import numpy as np
import onnxruntime as ort
from fastapi import FastAPI
from pydantic import BaseModel
from skl2onnx import to_onnx
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Save the whole pipeline, so preprocessing travels with the model.
pipeline = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
joblib.dump(pipeline, "model.joblib")
model = joblib.load("model.joblib")  # only ever load files you trust

# ONNX: a portable file that runs without Python or scikit-learn.
onnx_model = to_onnx(pipeline, X_train[:1].astype(np.float32))
with open("model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())
session = ort.InferenceSession("model.onnx")
input_name = session.get_inputs()[0].name
print(session.run(None, {input_name: X_test[:3].astype(np.float32)})[0])

# An online API: load the model once, then answer each request.
app = FastAPI()

class Features(BaseModel):
    values: list[float]

@app.post("/predict")
def predict(features: Features):
    probability = model.predict_proba([features.values])[0, 1]
    return {"label": int(probability > 0.5), "probability": float(probability)}

## Try it

In [ ]:
from fastapi.testclient import TestClient

# Send the service one request, as another program would over HTTP.
client = TestClient(app)
print(client.post("/predict", json={"values": X_test[0].tolist()}).json())